In [1]:
import os



# Define the path to the dataset
dataset_path = 'D:/MVSA_SINGLE'

# Check if the path exists and list the first few files
if os.path.exists(dataset_path):
    print(f"Successfully accessed: {dataset_path}")
    files = os.listdir(dataset_path)
    print(f"Total items found: {len(files)}")
    print("Sample files:", files[:5])
else:
    print(f"Path not found: {dataset_path}. Please ensure the folder name is correct.")

Successfully accessed: D:/MVSA_SINGLE
Total items found: 9
Sample files: ['bert_lstm_checkpoints', 'bert_lstm_final', 'data', 'labelResultAll.txt', 'mvsa_single_processed_224.parquet']


In [2]:
# =====================================================
# Cell - Check Parquet Dataset
# =====================================================
import pandas as pd
import os
import sys
import warnings

try:
    parquet_path = CONFIG.get('parquet_path', 'D:/MVSA_SINGLE/mvsa_single_processed_224.parquet')
except NameError:
    parquet_path = 'D:/MVSA_SINGLE/mvsa_single_processed_224.parquet'

if not os.path.exists(parquet_path):
    msg = f"Parquet file {parquet_path} not found. Please run 'mvsa-to-parquet.ipynb' first to generate the dataset."
    warnings.warn(msg, UserWarning)
    sys.exit(f"FileNotFoundError: {msg}")
else:
    print(f"File parquet found, loading from: {parquet_path}")
    # Optional preview load
    df = pd.read_parquet(parquet_path)
    print(f"Total entries loaded: {len(df)}")
    if 'final_sentiment' in df.columns:
        print("\nSentiment Distribution:")
        print(df['final_sentiment'].value_counts())
    display(df.head())


File parquet found, loading from: D:/MVSA_SINGLE/mvsa_single_processed_224.parquet
Total entries loaded: 4511

Sentiment Distribution:
final_sentiment
positive    2683
negative    1358
neutral      470
Name: count, dtype: int64


,ID,text_sentiment,image_sentiment,final_sentiment,image_bytes,text_content
0,1,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,How I feel today #legday #jelly #aching #gym
1,2,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,grattis min griskulting!!!???? va bara tvungen...
2,3,neutral,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,RT @polynminion: The moment I found my favouri...
3,4,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,#escort We have a young and energetic team and...
4,5,positive,positive,positive,b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\...,"RT @chrisashaffer: Went to SSC today to be a ""..."


In [3]:
# ─────────────────────────────────────────────
# Cell 1 — Install & Import Dependencies
# ─────────────────────────────────────────────
%pip install torch torchvision scikit-learn pyarrow tqdm gdown timm -q

import os, io, gc, re, json, random, warnings
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import timm
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print("All libraries imported successfully.")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"timm version    : {timm.__version__}")

Note: you may need to restart the kernel to use updated packages.


c:\Users\Residensi ADW\anaconda3\envs\env_gpu_pytorch_erl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully.
PyTorch version : 2.5.1
CUDA available  : True
GPU             : NVIDIA GeForce RTX 3050
GPU Memory      : 6.4 GB
timm version    : 1.0.26


In [4]:
# ─────────────────────────────────────────────
# Cell 2 — Model Configuration & Seed
# ─────────────────────────────────────────────
CONFIG = {
    # Paths
    "parquet_path": "D:/MVSA_SINGLE/mvsa_single_processed_224.parquet",
    "checkpoint_dir": "D:/MVSA_SINGLE/vit_lstm_checkpoints",
    "save_dir": "D:/MVSA_SINGLE/vit_lstm_final",

    # Text encoder: Embedding + LSTM
    "embed_dim": 300,
    "min_word_freq": 2,
    "max_seq_len": 64,
    "text_hidden_dim": 256,
    "text_num_layers": 2,
    "text_bidirectional": False,
    "text_dropout": 0.10,

    # Image encoder: ViT
    "image_backbone": "vit_base_patch16_224",
    "image_input_size": 224,
    "vit_drop_cls_token": True,
    "vit_dropout": 0.10,
    "vit_trainable_blocks": 4,

    # Shared multimodal dimension
    "d_model": 256,

    # Co-attention
    "co_attn_enabled": True,
    "co_attn_type": "multi-head",
    "co_attn_heads": 8,
    "co_attn_layers": 3,
    "co_attn_ffn_dim": 1024,
    "co_attn_dropout": 0.10,

    # Fusion and classifier
    "fusion_strategy": "concat_mean_pool",
    "classifier_hidden": 256,
    "classifier_dropout": 0.25,
    "num_classes": 3,

    # Training
    "epochs": 30,
    "batch_size": 16,
    "lr_main": 1e-4,
    "lr_vit": 1e-5,
    "weight_decay": 0.01,
    "gradient_clip": 1.0,
    "early_stop_patience": 7,
    "scheduler": "CosineAnnealingLR",

    # Data split
    "train_ratio": 0.80,
    "val_ratio": 0.10,
    "test_ratio": 0.10,

    # Runtime
    "seed": 42,
    "num_workers": 0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

assert abs(CONFIG["train_ratio"] + CONFIG["val_ratio"] + CONFIG["test_ratio"] - 1.0) < 1e-8
assert CONFIG["co_attn_type"] in {"single-head", "multi-head"}
assert CONFIG["d_model"] == CONFIG["text_hidden_dim"] * (2 if CONFIG["text_bidirectional"] else 1), (
    "d_model must match LSTM output dimension."
)
if CONFIG["co_attn_type"] == "single-head":
    CONFIG["co_attn_heads"] = 1
assert CONFIG["d_model"] % CONFIG["co_attn_heads"] == 0, "d_model must be divisible by attention heads"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CONFIG["seed"])
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
os.makedirs(CONFIG["save_dir"], exist_ok=True)

def print_config(config):
    sep = "=" * 74
    text_output_dim = config["text_hidden_dim"] * (2 if config["text_bidirectional"] else 1)
    vit_tuning = (
        f"last {config['vit_trainable_blocks']} block(s)"
        if config["vit_trainable_blocks"] > 0 else "frozen"
    )

    print(sep)
    print("  ViT + LSTM + Co-Attention Sentiment Analysis — Configuration")
    print(sep)
    print(f"  Device                      : {config['device'].upper()}")
    if torch.cuda.is_available():
        print(f"  GPU                         : {torch.cuda.get_device_name(0)}")
        print(f"  GPU Memory                  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    print("\n  ── Data ─────────────────────────────────────────────────────")
    print("  Source format               : Parquet")
    print(f"  Parquet path                : {config['parquet_path']}")
    print("  Fields used                 : text_content, image_bytes,")
    print("                                 text_sentiment, image_sentiment,")
    print("                                 final_sentiment (target label)")
    print(f"  Split (train/val/test)      : {config['train_ratio']:.2f}/{config['val_ratio']:.2f}/{config['test_ratio']:.2f}")
    print(f"  Image input size            : {config['image_input_size']}x{config['image_input_size']}")
    print(f"  Max text length             : {config['max_seq_len']}")
    print(f"  Min word frequency          : {config['min_word_freq']}")
    print(f"  DataLoader workers          : {config['num_workers']} (Colab-safe default)")

    print("\n  ── Text Encoder ─────────────────────────────────────────────")
    print("  Encoder                     : Trainable embedding + LSTM")
    print(f"  Embedding dim               : {config['embed_dim']}")
    print(f"  Hidden dim                  : {config['text_hidden_dim']}")
    print(f"  Bidirectional               : {config['text_bidirectional']}")
    print(f"  Number of LSTM layers       : {config['text_num_layers']}")
    print(f"  Text output dim             : {text_output_dim}")
    print(f"  Text dropout                : {config['text_dropout']}")

    print("\n  ── Image Encoder ────────────────────────────────────────────")
    print(f"  Backbone                    : {config['image_backbone']}")
    print("  Encoder                     : ViT patch tokens")
    print(f"  Drop CLS token              : {config['vit_drop_cls_token']}")
    print(f"  ViT dropout                 : {config['vit_dropout']}")
    print(f"  Fine-tuning policy          : {vit_tuning}")

    print("\n  ── Co-Attention ─────────────────────────────────────────────")
    print(f"  Enabled                     : {config['co_attn_enabled']}")
    print(f"  Attention style             : {config['co_attn_type']}")
    print(f"  Number of heads             : {config['co_attn_heads']}")
    print(f"  Number of co-attn layers    : {config['co_attn_layers']}")
    print(f"  Shared d_model              : {config['d_model']}")
    print(f"  FFN hidden dim              : {config['co_attn_ffn_dim']}")
    print(f"  Co-attn dropout             : {config['co_attn_dropout']}")

    print("\n  ── Classifier ───────────────────────────────────────────────")
    print(f"  Fusion strategy             : {config['fusion_strategy']}")
    print(f"  Classifier hidden dim       : {config['classifier_hidden']}")
    print(f"  Classifier dropout          : {config['classifier_dropout']}")
    print(f"  Number of classes           : {config['num_classes']}")

    print("\n  ── Training ─────────────────────────────────────────────────")
    print(f"  Epochs                      : {config['epochs']}")
    print(f"  Batch size                  : {config['batch_size']}")
    print(f"  Learning rate (new layers)  : {config['lr_main']}")
    print(f"  Learning rate (ViT)         : {config['lr_vit']}")
    print(f"  Weight decay                : {config['weight_decay']}")
    print(f"  Gradient clip               : {config['gradient_clip']}")
    print(f"  Scheduler                   : {config['scheduler']}")
    print(f"  Early-stop patience         : {config['early_stop_patience']}")
    print(f"  Random seed                 : {config['seed']}")
    print(sep)


print_config(CONFIG)

  ViT + LSTM + Co-Attention Sentiment Analysis — Configuration
  Device                      : CUDA
  GPU                         : NVIDIA GeForce RTX 3050
  GPU Memory                  : 6.4 GB

  ── Data ─────────────────────────────────────────────────────
  Source format               : Parquet
  Parquet path                : D:/MVSA_SINGLE/mvsa_single_processed_224.parquet
  Fields used                 : text_content, image_bytes,
                                 text_sentiment, image_sentiment,
                                 final_sentiment (target label)
  Split (train/val/test)      : 0.80/0.10/0.10
  Image input size            : 224x224
  Max text length             : 64
  Min word frequency          : 2
  DataLoader workers          : 0 (Colab-safe default)

  ── Text Encoder ─────────────────────────────────────────────
  Encoder                     : Trainable embedding + LSTM
  Embedding dim               : 300
  Hidden dim                  : 256
  Bidirectional        

In [5]:
# ─────────────────────────────────────────────
# Cell 3 — Load Parquet, Prepare Splits, Build Vocab, DataLoaders
# ─────────────────────────────────────────────
REQUIRED_COLUMNS = [
    "text_sentiment",
    "image_sentiment",
    "final_sentiment",
    "image_bytes",
    "text_content",
]
LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}
LABEL_IMAP = {v: k for k, v in LABEL_MAP.items()}
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"


def simple_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()


print(f"Loading parquet from: {CONFIG['parquet_path']}")
df_full = pd.read_parquet(CONFIG["parquet_path"])
print(f"Total rows loaded : {len(df_full)}")
print(f"Columns           : {list(df_full.columns)}")

missing_columns = [col for col in REQUIRED_COLUMNS if col not in df_full.columns]
if missing_columns:
    raise ValueError(f"Missing required parquet columns: {missing_columns}")

before = len(df_full)
df_full = df_full.dropna(subset=REQUIRED_COLUMNS).reset_index(drop=True)
print(f"Rows after dropna : {len(df_full)}  (dropped {before - len(df_full)})")

for col in ["text_sentiment", "image_sentiment", "final_sentiment"]:
    df_full[col] = df_full[col].astype(str).str.strip().str.lower()

df_full["label"] = df_full["final_sentiment"].map(LABEL_MAP)
if df_full["label"].isna().any():
    bad_count = int(df_full["label"].isna().sum())
    print(f"WARN: {bad_count} rows have invalid final_sentiment labels and will be dropped.")
    df_full = df_full.dropna(subset=["label"]).reset_index(drop=True)
df_full["label"] = df_full["label"].astype(int)

print("\nTarget distribution (final_sentiment):")
for label_name, label_idx in LABEL_MAP.items():
    count = int((df_full["label"] == label_idx).sum())
    pct = count / len(df_full) * 100
    print(f"  {label_idx}  {label_name:<10}  {count:>5}  ({pct:.1f}%)")

print("\nText modality sentiment distribution:")
print(df_full["text_sentiment"].value_counts().to_string())
print("\nImage modality sentiment distribution:")
print(df_full["image_sentiment"].value_counts().to_string())

df_train, df_tmp = train_test_split(
    df_full,
    test_size=(CONFIG["val_ratio"] + CONFIG["test_ratio"]),
    stratify=df_full["label"],
    random_state=CONFIG["seed"],
)
df_val, df_test = train_test_split(
    df_tmp,
    test_size=CONFIG["test_ratio"] / (CONFIG["val_ratio"] + CONFIG["test_ratio"]),
    stratify=df_tmp["label"],
    random_state=CONFIG["seed"],
)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

print("\nData split (stratified):")
print(f"  Train : {len(df_train):>5} rows")
print(f"  Val   : {len(df_val):>5} rows")
print(f"  Test  : {len(df_test):>5} rows")

counter = Counter()
for text in df_train["text_content"]:
    counter.update(simple_tokenize(text))

vocab = [PAD_TOKEN, UNK_TOKEN] + [
    word for word, freq in counter.items() if freq >= CONFIG["min_word_freq"]
]
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
VOCAB_SIZE = len(vocab)
PAD_IDX = word2idx[PAD_TOKEN]
UNK_IDX = word2idx[UNK_TOKEN]

embedding_matrix = np.random.normal(
    loc=0.0,
    scale=0.02,
    size=(VOCAB_SIZE, CONFIG["embed_dim"]),
).astype(np.float32)
embedding_matrix[PAD_IDX] = 0.0

print(f"\nVocabulary size              : {VOCAB_SIZE}")
print(f"Embedding matrix shape       : {embedding_matrix.shape}")
print(f"PAD index / UNK index        : {PAD_IDX} / {UNK_IDX}")
print("Text and image sentiment fields are loaded as metadata; final_sentiment is the target.")

IMAGE_MEAN = [0.485, 0.456, 0.406]
IMAGE_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_input_size"], CONFIG["image_input_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((CONFIG["image_input_size"], CONFIG["image_input_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD),
])


class MVSAMultimodalDataset(Dataset):
    def __init__(self, dataframe, word2idx, max_len, image_transform, pad_idx, unk_idx):
        self.df = dataframe.reset_index(drop=True)
        self.word2idx = word2idx
        self.max_len = max_len
        self.transform = image_transform
        self.pad_idx = pad_idx
        self.unk_idx = unk_idx

    def __len__(self):
        return len(self.df)

    def _encode_text(self, text):
        tokens = simple_tokenize(text)
        if not tokens:
            tokens = [UNK_TOKEN]
        tokens = tokens[: self.max_len]
        token_ids = [self.word2idx.get(token, self.unk_idx) for token in tokens]
        seq_len = len(token_ids)
        if seq_len < self.max_len:
            token_ids = token_ids + [self.pad_idx] * (self.max_len - seq_len)
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(seq_len, dtype=torch.long)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        raw_image = row["image_bytes"]
        raw_image = raw_image if isinstance(raw_image, bytes) else bytes(raw_image)
        image = Image.open(io.BytesIO(raw_image)).convert("RGB")
        image_tensor = self.transform(image)

        token_ids, seq_len = self._encode_text(row["text_content"])

        return {
            "image": image_tensor,
            "text": token_ids,
            "seq_len": seq_len,
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
            "text_sentiment": row["text_sentiment"],
            "image_sentiment": row["image_sentiment"],
            "final_sentiment": row["final_sentiment"],
        }


train_dataset = MVSAMultimodalDataset(
    df_train,
    word2idx=word2idx,
    max_len=CONFIG["max_seq_len"],
    image_transform=train_transform,
    pad_idx=PAD_IDX,
    unk_idx=UNK_IDX,
)
val_dataset = MVSAMultimodalDataset(
    df_val,
    word2idx=word2idx,
    max_len=CONFIG["max_seq_len"],
    image_transform=eval_transform,
    pad_idx=PAD_IDX,
    unk_idx=UNK_IDX,
)
test_dataset = MVSAMultimodalDataset(
    df_test,
    word2idx=word2idx,
    max_len=CONFIG["max_seq_len"],
    image_transform=eval_transform,
    pad_idx=PAD_IDX,
    unk_idx=UNK_IDX,
)

loader_kwargs = {
    "batch_size": CONFIG["batch_size"],
    "num_workers": CONFIG["num_workers"],
    "pin_memory": torch.cuda.is_available(),
}
train_loader = DataLoader(train_dataset, shuffle=True, drop_last=False, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, drop_last=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, drop_last=False, **loader_kwargs)

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(CONFIG["num_classes"]),
    y=df_train["label"].values,
)

print(f"Train batches : {len(train_loader):>4}  ({len(train_dataset)} samples)")
print(f"Val   batches : {len(val_loader):>4}  ({len(val_dataset)} samples)")
print(f"Test  batches : {len(test_loader):>4}  ({len(test_dataset)} samples)")
print(
    "Class weights : "
    f"negative={class_weights_np[0]:.3f}  neutral={class_weights_np[1]:.3f}  positive={class_weights_np[2]:.3f}"
)

sample_batch = next(iter(train_loader))
print("\nBatch shapes:")
print(f"  image         : {tuple(sample_batch['image'].shape)}")
print(f"  text          : {tuple(sample_batch['text'].shape)}")
print(f"  seq_len       : {tuple(sample_batch['seq_len'].shape)}  sample={sample_batch['seq_len'][:4].tolist()}")
print(f"  label         : {tuple(sample_batch['label'].shape)}  sample={sample_batch['label'][:4].tolist()}")
print(f"  text_sentiment: {sample_batch['text_sentiment'][:3]}")
print(f"  image_sentiment: {sample_batch['image_sentiment'][:3]}")
del sample_batch

Loading parquet from: D:/MVSA_SINGLE/mvsa_single_processed_224.parquet
Total rows loaded : 4511
Columns           : ['ID', 'text_sentiment', 'image_sentiment', 'final_sentiment', 'image_bytes', 'text_content']
Rows after dropna : 4511  (dropped 0)

Target distribution (final_sentiment):
  0  negative     1358  (30.1%)
  1  neutral       470  (10.4%)
  2  positive     2683  (59.5%)

Text modality sentiment distribution:
text_sentiment
neutral     1921
positive    1653
negative     937

Image modality sentiment distribution:
image_sentiment
positive    2428
negative    1145
neutral      938

Data split (stratified):
  Train :  3608 rows
  Val   :   451 rows
  Test  :   452 rows

Vocabulary size              : 4192
Embedding matrix shape       : (4192, 300)
PAD index / UNK index        : 0 / 1
Text and image sentiment fields are loaded as metadata; final_sentiment is the target.
Train batches :  226  (3608 samples)
Val   batches :   29  (451 samples)
Test  batches :   29  (452 samples)
Cl

In [6]:
# ─────────────────────────────────────────────
# Cell 4 — ViT + LSTM + Co-Attention Model
# ─────────────────────────────────────────────
class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, pad_idx, pretrained_weights, bidirectional=False, dropout=0.3, out_dim=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embedding.weight = nn.Parameter(torch.from_numpy(pretrained_weights), requires_grad=True)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        lstm_out_dim = hidden_dim * (2 if bidirectional else 1)
        self.output_proj = nn.Identity() if out_dim is None or lstm_out_dim == out_dim else nn.Linear(lstm_out_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, token_ids, seq_lens):
        embedded = self.embedding(token_ids)
        packed = pack_padded_sequence(
            embedded,
            lengths=seq_lens.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_out, _ = self.lstm(packed)
        text_out, _ = pad_packed_sequence(
            packed_out,
            batch_first=True,
            total_length=token_ids.size(1),
        )
        text_out = self.output_proj(text_out)
        return self.dropout(text_out)


class ViTImageEncoder(nn.Module):
    def __init__(self, backbone_name, out_dim, drop_cls_token=True, dropout=0.1, trainable_blocks=4):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0, global_pool="")
        self.drop_cls_token = drop_cls_token
        self.proj = nn.Sequential(
            nn.Linear(self.backbone.num_features, out_dim),
            nn.LayerNorm(out_dim),
        )
        self.dropout = nn.Dropout(dropout)
        self._freeze_backbone(trainable_blocks)

    def _freeze_backbone(self, trainable_blocks):
        for param in self.backbone.parameters():
            param.requires_grad = False

        if trainable_blocks <= 0:
            return

        if hasattr(self.backbone, "blocks"):
            for block in self.backbone.blocks[-trainable_blocks:]:
                for param in block.parameters():
                    param.requires_grad = True
        if hasattr(self.backbone, "norm"):
            for param in self.backbone.norm.parameters():
                param.requires_grad = True
        for name in ["cls_token", "pos_embed"]:
            if hasattr(self.backbone, name):
                attr = getattr(self.backbone, name)
                if isinstance(attr, torch.nn.Parameter):
                    attr.requires_grad = True

    def forward(self, pixel_values):
        image_tokens = self.backbone.forward_features(pixel_values)
        if isinstance(image_tokens, (list, tuple)):
            image_tokens = image_tokens[-1]
        if image_tokens.ndim == 2:
            image_tokens = image_tokens.unsqueeze(1)
        if self.drop_cls_token and image_tokens.size(1) > 1:
            image_tokens = image_tokens[:, 1:, :]
        image_tokens = self.proj(image_tokens)
        image_tokens = self.dropout(image_tokens)
        return image_tokens


class CoAttentionLayer(nn.Module):
    def __init__(self, d_model, num_heads, ffn_dim, dropout):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.text_to_image = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.image_to_text = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.text_norm1 = nn.LayerNorm(d_model)
        self.text_norm2 = nn.LayerNorm(d_model)
        self.image_norm1 = nn.LayerNorm(d_model)
        self.image_norm2 = nn.LayerNorm(d_model)
        self.text_ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.image_ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, text_feats, image_feats, text_pad_mask=None):
        if text_pad_mask is not None:
            all_masked = text_pad_mask.all(dim=1)
            if all_masked.any():
                text_pad_mask = text_pad_mask.clone()
                text_pad_mask[all_masked] = False

        text_attended, _ = self.text_to_image(
            query=text_feats,
            key=image_feats,
            value=image_feats,
        )
        text_feats = self.text_norm1(text_feats + text_attended)
        text_feats = self.text_norm2(text_feats + self.text_ffn(text_feats))

        image_attended, _ = self.image_to_text(
            query=image_feats,
            key=text_feats,
            value=text_feats,
            key_padding_mask=text_pad_mask,
        )
        image_feats = self.image_norm1(image_feats + image_attended)
        image_feats = self.image_norm2(image_feats + self.image_ffn(image_feats))
        return text_feats, image_feats


class MultimodalSentimentModel(nn.Module):
    def __init__(self, config, vocab_size, pad_idx, pretrained_weights):
        super().__init__()
        self.pad_idx = pad_idx
        self.co_attn_enabled = config["co_attn_enabled"]
        self.text_encoder = TextEncoder(
            vocab_size=vocab_size,
            embed_dim=config["embed_dim"],
            hidden_dim=config["text_hidden_dim"],
            num_layers=config["text_num_layers"],
            pad_idx=pad_idx,
            pretrained_weights=pretrained_weights,
            bidirectional=config["text_bidirectional"],
            dropout=config["text_dropout"],
            out_dim=config["d_model"],
        )
        self.image_encoder = ViTImageEncoder(
            backbone_name=config["image_backbone"],
            out_dim=config["d_model"],
            drop_cls_token=config["vit_drop_cls_token"],
            dropout=config["vit_dropout"],
            trainable_blocks=config["vit_trainable_blocks"],
        )
        self.co_attention_layers = nn.ModuleList([
            CoAttentionLayer(
                d_model=config["d_model"],
                num_heads=config["co_attn_heads"],
                ffn_dim=config["co_attn_ffn_dim"],
                dropout=config["co_attn_dropout"],
            )
            for _ in range(config["co_attn_layers"])
        ])
        fusion_dim = config["d_model"] * 2
        self.classifier = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Linear(fusion_dim, config["classifier_hidden"]),
            nn.GELU(),
            nn.Dropout(config["classifier_dropout"]),
            nn.Linear(config["classifier_hidden"], config["num_classes"]),
        )

    def forward(self, text_ids, seq_lens, images):
        text_pad_mask = text_ids.eq(self.pad_idx)
        text_feats = self.text_encoder(text_ids, seq_lens)
        image_feats = self.image_encoder(images)

        if self.co_attn_enabled:
            for layer in self.co_attention_layers:
                text_feats, image_feats = layer(
                    text_feats,
                    image_feats,
                    text_pad_mask=text_pad_mask,
                )

        valid_mask = (~text_pad_mask).float().unsqueeze(-1)
        text_pool = (text_feats * valid_mask).sum(dim=1) / valid_mask.sum(dim=1).clamp(min=1.0)
        image_pool = image_feats.mean(dim=1)
        fused = torch.cat([text_pool, image_pool], dim=-1)
        logits = self.classifier(fused)
        return logits


device = torch.device(CONFIG["device"])
model = MultimodalSentimentModel(
    config=CONFIG,
    vocab_size=VOCAB_SIZE,
    pad_idx=PAD_IDX,
    pretrained_weights=embedding_matrix,
).to(device)

total_params = sum(param.numel() for param in model.parameters())
trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
vit_trainable_params = sum(param.numel() for param in model.image_encoder.backbone.parameters() if param.requires_grad)
print(f"Total parameters         : {total_params:,}")
print(f"Trainable parameters     : {trainable_params:,}")
print(f"Trainable ViT parameters : {vit_trainable_params:,}")

model.eval()
with torch.no_grad():
    dummy_batch = next(iter(train_loader))
    dummy_logits = model(
        dummy_batch["text"].to(device),
        dummy_batch["seq_len"].to(device),
        dummy_batch["image"].to(device),
    )
print(f"Forward pass OK  |  logits shape: {tuple(dummy_logits.shape)}")
del dummy_batch, dummy_logits
gc.collect()
model.train()

Total parameters         : 93,223,043
Trainable parameters     : 35,929,475
Trainable ViT parameters : 28,505,088
Forward pass OK  |  logits shape: (16, 3)


MultimodalSentimentModel(
  (text_encoder): TextEncoder(
    (embedding): Embedding(4192, 300, padding_idx=0)
    (lstm): LSTM(300, 256, num_layers=2, batch_first=True, dropout=0.1)
    (output_proj): Identity()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (image_encoder): ViTImageEncoder(
    (backbone): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (proj): Linear(in_features=768, out

In [7]:
# ─────────────────────────────────────────────
# Cell 5 — Training Loop, Evaluation, Save Artifacts
# ─────────────────────────────────────────────
vit_param_ids = {
    id(param)
    for param in model.image_encoder.backbone.parameters()
    if param.requires_grad
}
vit_params = [param for param in model.parameters() if id(param) in vit_param_ids and param.requires_grad]
main_params = [param for param in model.parameters() if id(param) not in vit_param_ids and param.requires_grad]

optimizer_groups = [{"params": main_params, "lr": CONFIG["lr_main"]}]
if vit_params:
    optimizer_groups.append({"params": vit_params, "lr": CONFIG["lr_vit"]})

optimizer = AdamW(optimizer_groups, weight_decay=CONFIG["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"], eta_min=1e-6)
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print("Optimizer groups:")
print(f"  New/trainable non-ViT params : {len(main_params)} tensors  lr={CONFIG['lr_main']}")
print(f"  ViT params                   : {len(vit_params)} tensors  lr={CONFIG['lr_vit']}")


def evaluate(loader, model, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    nan_batches = 0

    with torch.no_grad():
        for batch in loader:
            text = batch["text"].to(device)
            seq_len = batch["seq_len"].to(device)
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            logits = model(text, seq_len, images)
            loss = criterion(logits, labels)
            if torch.isnan(loss):
                nan_batches += 1
                continue

            total_loss += loss.item()
            all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    valid_batches = len(loader) - nan_batches
    if valid_batches <= 0:
        return float("nan"), 0.0, 0.0

    avg_loss = total_loss / valid_batches
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, f1


best_val_f1 = 0.0
patience_counter = 0
best_ckpt_path = os.path.join(CONFIG["checkpoint_dir"], "vit_lstm_best.pt")
history = []
nan_batches_total = 0

header = (
    f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  "
    f"{'Val Loss':>8}  {'Val Acc':>7}  {'Val F1':>7}  {'LR(main)':>10}  {'Status'}"
)
print(header)
print("─" * len(header))

for epoch in range(1, CONFIG["epochs"] + 1):
    model.train()
    running_loss = 0.0
    train_preds, train_labels = [], []
    nan_this_epoch = 0

    progress = tqdm(train_loader, desc=f"Epoch {epoch:>2}/{CONFIG['epochs']}", leave=False)
    for batch in progress:
        text = batch["text"].to(device)
        seq_len = batch["seq_len"].to(device)
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(text, seq_len, images)
        loss = criterion(logits, labels)

        if torch.isnan(loss):
            nan_this_epoch += 1
            nan_batches_total += 1
            progress.set_postfix(loss="NaN-skip")
            continue

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip"])
        optimizer.step()

        running_loss += loss.item()
        train_preds.extend(logits.detach().argmax(dim=-1).cpu().tolist())
        train_labels.extend(labels.cpu().tolist())
        progress.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()

    valid_train_batches = len(train_loader) - nan_this_epoch
    train_loss = running_loss / valid_train_batches if valid_train_batches > 0 else float("nan")
    train_acc = accuracy_score(train_labels, train_preds) if train_labels else 0.0
    val_loss, val_acc, val_f1 = evaluate(val_loader, model, criterion, device)
    lr_main = optimizer.param_groups[0]["lr"]

    status = ""
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "val_f1": val_f1,
                "val_acc": val_acc,
                "config": CONFIG,
                "word2idx": word2idx,
                "pad_idx": PAD_IDX,
                "unk_idx": UNK_IDX,
            },
            best_ckpt_path,
        )
        status = "✓ best saved"
    else:
        patience_counter += 1
        status = f"patience {patience_counter}/{CONFIG['early_stop_patience']}"
        if patience_counter >= CONFIG["early_stop_patience"]:
            print(
                f"\nEarly stopping at epoch {epoch} "
                f"(no improvement for {CONFIG['early_stop_patience']} epochs)"
            )
            break

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
        }
    )

    if nan_this_epoch:
        print(f"  [WARN] epoch {epoch}: {nan_this_epoch} NaN-loss batches skipped.")

    print(
        f"{epoch:>5}  {train_loss:>10.4f}  {train_acc:>9.4f}  "
        f"{val_loss:>8.4f}  {val_acc:>7.4f}  {val_f1:>7.4f}  {lr_main:>10.2e}  {status}"
    )

print(f"\nTraining complete. Best validation macro-F1: {best_val_f1:.4f}")
print(f"Best checkpoint path: {best_ckpt_path}")
if nan_batches_total:
    print(f"Total NaN-loss batches skipped: {nan_batches_total}")

print(f"Loading best checkpoint: {best_ckpt_path}")
checkpoint = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(checkpoint["model_state"])
print(
    f"Checkpoint epoch: {checkpoint['epoch']}  "
    f"val_f1={checkpoint['val_f1']:.4f}  val_acc={checkpoint['val_acc']:.4f}"
)

model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        text = batch["text"].to(device)
        seq_len = batch["seq_len"].to(device)
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        logits = model(text, seq_len, images)
        probs = F.softmax(logits, dim=-1)
        preds = logits.argmax(dim=-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())

class_names = [LABEL_IMAP[idx] for idx in range(CONFIG["num_classes"])]
test_acc = accuracy_score(all_labels, all_preds)
test_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
cm = confusion_matrix(all_labels, all_preds)

sep = "=" * 74
print(f"\n{sep}")
print("  Test Set Results — ViT + LSTM + Co-Attention")
print(sep)
print(f"  Accuracy  (overall)        : {test_acc:.4f}  ({test_acc * 100:.2f}%)")
print(f"  Macro F1  (unweighted)     : {test_f1:.4f}")
print(f"  Attention type used        : {CONFIG['co_attn_type']} ({CONFIG['co_attn_heads']} head(s))")
print(f"  Epochs configured          : {CONFIG['epochs']}")
print(f"  Batch size                 : {CONFIG['batch_size']}")
print(f"  Learning rate (new layers) : {CONFIG['lr_main']}")
print(f"  Learning rate (ViT)        : {CONFIG['lr_vit']}")
print("\n  Classification report:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4, zero_division=0))

print("  Confusion Matrix (rows=true, cols=pred):")
col_width = max(len(name) for name in class_names) + 2
print(" " * 12 + "  ".join(f"{name:>{col_width}}" for name in class_names))
for idx, row_values in enumerate(cm):
    row_str = "  ".join(f"{value:{col_width}d}" for value in row_values)
    print(f"  {class_names[idx]:>10}  {row_str}")

if history:
    history_df = pd.DataFrame(history)
    print("\n  Training history (last 5 epochs):")
    print(history_df.tail(5).to_string(index=False))

artifact_path = os.path.join(CONFIG["save_dir"], "vit_lstm_full.pt")
weights_path = os.path.join(CONFIG["save_dir"], "vit_lstm_weights_only.pt")
config_path = os.path.join(CONFIG["save_dir"], "vit_lstm_config.json")

os.makedirs(CONFIG["save_dir"], exist_ok=True)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "config": CONFIG,
        "label_map": LABEL_MAP,
        "label_imap": LABEL_IMAP,
        "word2idx": word2idx,
        "idx2word": idx2word,
        "pad_idx": PAD_IDX,
        "unk_idx": UNK_IDX,
        "best_val_f1": best_val_f1,
        "test_acc": test_acc,
        "test_f1": test_f1,
        "history": history,
    },
    artifact_path,
)
torch.save(model.state_dict(), weights_path)

config_to_save = {key: value for key, value in CONFIG.items() if not callable(value)}
config_to_save["label_map"] = LABEL_MAP
config_to_save["label_imap"] = {str(key): value for key, value in LABEL_IMAP.items()}
config_to_save["vocab_size"] = VOCAB_SIZE
with open(config_path, "w") as file:
    json.dump(config_to_save, file, indent=2)

print(f"\nArtifacts saved to: {CONFIG['save_dir']}")
for path in [artifact_path, weights_path, config_path]:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  {os.path.basename(path):<32} {size_mb:.2f} MB")
print(sep)

Optimizer groups:
  New/trainable non-ViT params : 91 tensors  lr=0.0001
  ViT params                   : 52 tensors  lr=1e-05
Epoch  Train Loss  Train Acc  Val Loss  Val Acc   Val F1    LR(main)  Status
────────────────────────────────────────────────────────────────────────────


    1      0.9887     0.5516    0.9353   0.6364   0.5560    9.97e-05  ✓ best saved


    2      0.7247     0.7015    0.9334   0.6009   0.5484    9.89e-05  patience 1/7


    3      0.5112     0.7913    1.1464   0.6541   0.5729    9.76e-05  ✓ best saved


    4      0.3221     0.8744    1.4721   0.6541   0.5615    9.57e-05  patience 1/7


...

   19      0.0221     0.9920    3.7846   0.6718   0.5748    3.04e-05  patience 5/7


   20      0.0190     0.9914    3.7027   0.6696   0.5758    2.58e-05  patience 6/7



Early stopping at epoch 21 (no improvement for 7 epochs)

Training complete. Best validation macro-F1: 0.5784
Best checkpoint path: D:/MVSA_SINGLE/vit_lstm_checkpoints\vit_lstm_best.pt
Loading best checkpoint: D:/MVSA_SINGLE/vit_lstm_checkpoints\vit_lstm_best.pt
Checkpoint epoch: 14  val_f1=0.5784  val_acc=0.6763


Testing: 100%|██████████| 29/29 [00:14<00:00,  1.96it/s]



  Test Set Results — ViT + LSTM + Co-Attention
  Accuracy  (overall)        : 0.6504  (65.04%)
  Macro F1  (unweighted)     : 0.5359
  Attention type used        : multi-head (8 head(s))
  Epochs configured          : 30
  Batch size                 : 16
  Learning rate (new layers) : 0.0001
  Learning rate (ViT)        : 1e-05

  Classification report:
              precision    recall  f1-score   support

    negative     0.5929    0.4926    0.5382       136
     neutral     0.3415    0.2979    0.3182        47
    positive     0.7148    0.7918    0.7513       269

    accuracy                         0.6504       452
   macro avg     0.5497    0.5274    0.5359       452
weighted avg     0.6393    0.6504    0.6421       452

  Confusion Matrix (rows=true, cols=pred):
              negative     neutral    positive
    negative          67          11          58
     neutral           6          14          27
    positive          40          16         213

  Training history (last